In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from supabase import create_client
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
from scipy import stats
import random
from datetime import datetime, timedelta

load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_SERVICE_ROLE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

print("✅ Conectado a Supabase")

✅ Conectado a Supabase


In [22]:
def fetch_all_data(table_name, batch_size=5000):
    all_data = []
    offset = 0
    while True:
        response = supabase.table(table_name).select('*').order('timestamp').range(offset, offset+batch_size-1).execute()
        if not response.data:
            break
        all_data.extend(response.data)
        offset += batch_size
        print(f"Descargadas {len(all_data)} filas...")
    df = pd.DataFrame(all_data)
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
    return df

df_full = fetch_all_data('nq_5min')
print(f"✅ Datos cargados: {len(df_full):,} velas de 5 minutos")
print(f"Rango completo: {df_full['timestamp'].min()} → {df_full['timestamp'].max()}")

Descargadas 1000 filas...
Descargadas 2000 filas...
Descargadas 3000 filas...
Descargadas 4000 filas...
Descargadas 5000 filas...
Descargadas 6000 filas...
Descargadas 7000 filas...
Descargadas 8000 filas...
Descargadas 9000 filas...
Descargadas 10000 filas...
Descargadas 11000 filas...
Descargadas 12000 filas...
Descargadas 13000 filas...
Descargadas 14000 filas...
Descargadas 15000 filas...
Descargadas 16000 filas...
Descargadas 17000 filas...
Descargadas 18000 filas...
Descargadas 19000 filas...
Descargadas 20000 filas...
Descargadas 21000 filas...
Descargadas 22000 filas...
Descargadas 23000 filas...
Descargadas 24000 filas...
Descargadas 25000 filas...
Descargadas 26000 filas...
Descargadas 27000 filas...
Descargadas 28000 filas...
Descargadas 29000 filas...
Descargadas 30000 filas...
Descargadas 31000 filas...
Descargadas 32000 filas...
Descargadas 33000 filas...
Descargadas 34000 filas...
Descargadas 35000 filas...
Descargadas 36000 filas...
Descargadas 37000 filas...
Descargada

In [23]:
df_full = df_full.set_index('timestamp')
df_full = df_full.between_time('09:30', '16:00')  # Regular Trading Hours
print(f"Velas dentro de RTH: {len(df_full)}")

def add_atr(df, period=14):
    df['high_low'] = df['high'] - df['low']
    df['high_close'] = abs(df['high'] - df['close'].shift(1))
    df['low_close'] = abs(df['low'] - df['close'].shift(1))
    df['tr'] = df[['high_low', 'high_close', 'low_close']].max(axis=1)
    df['atr'] = df['tr'].rolling(window=period).mean()
    return df

df_full = add_atr(df_full, period=14)
df_full = df_full.dropna().copy()
print(f"Velas procesables con ATR: {len(df_full)}")

Velas dentro de RTH: 24988
Velas procesables con ATR: 24975


In [24]:
def find_swing_levels(df, window=20):
    highs = df['high'].values
    lows = df['low'].values
    timestamps = df.index
    levels = []
    for i in range(window, len(df) - window):
        if highs[i] == max(highs[i-window:i+window+1]) and highs[i] > highs[i-1] and highs[i] > highs[i+1]:
            levels.append((timestamps[i], highs[i], 'resistance'))
        if lows[i] == min(lows[i-window:i+window+1]) and lows[i] < lows[i-1] and lows[i] < lows[i+1]:
            levels.append((timestamps[i], lows[i], 'support'))
    return levels

# Precalcular niveles globales (sin look‑ahead) para uso en backtest
swing_levels = find_swing_levels(df_full, window=20)
swing_df = pd.DataFrame(swing_levels, columns=['timestamp', 'level', 'type']).set_index('timestamp')
print(f"Niveles encontrados: {len(swing_df)}")

Niveles encontrados: 749


In [25]:
def detect_fade_support(df, idx, swing_df, distance_pct=0.0008, min_age_minutes=30):
    if idx < 20:
        return None
    current_time = df.index[idx]
    past = swing_df[swing_df.index < current_time]
    if past.empty:
        return None
    current_open = df['open'].iloc[idx]
    for t, row in past.iterrows():
        if (current_time - t).total_seconds() < min_age_minutes * 60:
            continue
        if row['type'] == 'support' and abs(current_open - row['level']) / row['level'] < distance_pct and current_open <= row['level']:
            return ('BUY', current_open, 'fade_support')
    return None

def detect_range_breakout(df, idx, lookback=12, range_percentile=0.2):
    if idx < lookback:
        return None
    segment = df.iloc[idx-lookback:idx]
    range_width = segment['high'] - segment['low']
    threshold = range_width.quantile(range_percentile)
    if (segment['high'].max() - segment['low'].min()) > threshold:
        return None
    high_band = segment['high'].max()
    low_band = segment['low'].min()
    current = df.iloc[idx]
    if current['open'] > high_band:
        return ('BUY', current['open'], 'breakout_long')
    elif current['open'] < low_band:
        return ('SELL', current['open'], 'breakout_short')
    return None

In [26]:
def run_backtest(df, swing_df, params):
    trades = []
    position = None
    commission = 0.5   # $0.50 por operación (ida y vuelta)
    slippage = 0.25    # 0.25 puntos por lado

    for i in range(1, len(df)):
        current_time = df.index[i]
        open_price = df['open'].iloc[i]
        high = df['high'].iloc[i]
        low = df['low'].iloc[i]
        close = df['close'].iloc[i]
        atr = df['atr'].iloc[i]

        if pd.isna(atr) or atr < params['min_atr']:
            continue

        if position is None:
            signal = detect_fade_support(df, i, swing_df, distance_pct=params['distance_pct'])
            if not signal:
                signal = detect_range_breakout(df, i, lookback=params['lookback'], range_percentile=params['range_percentile'])
            if signal:
                direction, entry_price, setup = signal
                entry_price_with_slippage = entry_price + slippage if direction == 'BUY' else entry_price - slippage
                if setup == 'fade_support':
                    stop = entry_price_with_slippage - params['stop_fade'] * atr
                    target = entry_price_with_slippage + params['target_fade'] * atr
                else:  # breakout
                    stop = entry_price_with_slippage - params['stop_breakout'] * atr
                    target = entry_price_with_slippage + params['target_breakout'] * atr
                position = {
                    'direction': direction,
                    'entry_price': entry_price_with_slippage,
                    'entry_time': current_time,
                    'stop': stop,
                    'target': target,
                    'setup': setup
                }
        else:
            exit_price = None
            if position['direction'] == 'BUY':
                if high >= position['target']:
                    exit_price = position['target']
                elif low <= position['stop']:
                    exit_price = position['stop']
            else:
                if low <= position['target']:
                    exit_price = position['target']
                elif high >= position['stop']:
                    exit_price = position['stop']

            if current_time.hour == 15 and current_time.minute >= 45:  # forced close 15:45
                exit_price = close

            if exit_price is not None:
                if position['direction'] == 'BUY':
                    pnl_points = exit_price - position['entry_price']
                else:
                    pnl_points = position['entry_price'] - exit_price
                pnl_points -= commission / 5   # comisión en puntos (aprox. $0.50 = 0.1 puntos? Ajusta)
                trades.append({
                    'entry_time': position['entry_time'],
                    'exit_time': current_time,
                    'direction': position['direction'],
                    'entry_price': position['entry_price'],
                    'exit_price': exit_price,
                    'pnl_points': pnl_points,
                    'setup': position['setup']
                })
                position = None

    return pd.DataFrame(trades)

In [28]:
params_base = {
    'min_atr': 15,
    'distance_pct': 0.0008,
    'lookback': 12,
    'range_percentile': 0.2,
    'stop_fade': 0.2,
    'target_fade': 0.5,
    'stop_breakout': 0.3,
    'target_breakout': 1.0
}

trades = run_backtest(df_full, swing_df, params_base)
print(f"Backtest completado. Operaciones: {len(trades)}")

Backtest completado. Operaciones: 1826


In [29]:
if len(trades) > 0:
    trades['win'] = trades['pnl_points'] > 0
    win_rate = trades['win'].mean()
    avg_win = trades[trades['win']]['pnl_points'].mean() if win_rate > 0 else 0
    avg_loss = trades[~trades['win']]['pnl_points'].mean() if win_rate < 1 else 0
    profit_factor = (avg_win * win_rate) / (abs(avg_loss) * (1 - win_rate)) if avg_loss != 0 else np.inf
    total_points = trades['pnl_points'].sum()
    
    # Sharpe aproximado (diario)
    daily_pnl = trades.resample('D', on='entry_time')['pnl_points'].sum()
    sharpe = daily_pnl.mean() / daily_pnl.std() * np.sqrt(252) if daily_pnl.std() != 0 else 0
    
    # Drawdown realista (sobre capital, sin valores negativos)
    capital_inicial = 25000
    valor_punto = 5          # 1 punto en MNQ = $5
    trades['pnl_usd'] = trades['pnl_points'] * valor_punto
    trades['equity_usd'] = capital_inicial + trades['pnl_usd'].cumsum()
    trades['equity_usd'] = trades['equity_usd'].clip(lower=0)   # no puede ser negativo
    rolling_max = trades['equity_usd'].expanding().max()
    trades['drawdown_usd'] = (trades['equity_usd'] - rolling_max) / rolling_max * 100
    trades['drawdown_usd'] = trades['drawdown_usd'].fillna(0)
    max_dd = trades['drawdown_usd'].min()

    print("=== BACKTEST METRICS ===")
    print(f"Total trades: {len(trades)}")
    print(f"Win rate: {win_rate:.1%}")
    print(f"Avg win: {avg_win:.2f} pts")
    print(f"Avg loss: {avg_loss:.2f} pts")
    print(f"Profit factor: {profit_factor:.2f}")
    print(f"Total points: {total_points:.1f}")
    print(f"Sharpe ratio (approx): {sharpe:.2f}")
    print(f"Max drawdown: {max_dd:.2f}%")
    print(f"\nWith 3 MNQ contracts: total PnL = ${total_points * 15:,.0f}")
else:
    print("No trades generated. Adjust parameters.")

=== BACKTEST METRICS ===
Total trades: 1826
Win rate: 49.5%
Avg win: 16.56 pts
Avg loss: -7.79 pts
Profit factor: 2.08
Total points: 7767.0
Sharpe ratio (approx): 2.85
Max drawdown: -1.79%

With 3 MNQ contracts: total PnL = $116,505


In [30]:
# Walk‑forward: In‑Sample (2020-2023) vs Out‑of‑Sample (2024-2026)
df_full_reset = df_full.reset_index()
mask_is = df_full_reset['timestamp'] < '2024-01-01'
mask_oos = df_full_reset['timestamp'] >= '2024-01-01'

df_is = df_full_reset[mask_is].set_index('timestamp')
df_oos = df_full_reset[mask_oos].set_index('timestamp')

# Recalcular niveles dentro de cada conjunto (sin futuro)
swing_is = find_swing_levels(df_is, window=20)
swing_df_is = pd.DataFrame(swing_is, columns=['timestamp','level','type']).set_index('timestamp')
trades_is = run_backtest(df_is, swing_df_is, params_base)

swing_oos = find_swing_levels(df_oos, window=20)
swing_df_oos = pd.DataFrame(swing_oos, columns=['timestamp','level','type']).set_index('timestamp')
trades_oos = run_backtest(df_oos, swing_df_oos, params_base)

print("=== IN-SAMPLE (2020-2023) ===")
if len(trades_is) > 0:
    trades_is['win'] = trades_is['pnl_points'] > 0
    print(f"Trades: {len(trades_is)}, Win rate: {trades_is['win'].mean():.1%}")
else:
    print("No trades generados en in‑sample.")

print("\n=== OUT-OF-SAMPLE (2024-2026) ===")
if len(trades_oos) > 0:
    trades_oos['win'] = trades_oos['pnl_points'] > 0
    print(f"Trades: {len(trades_oos)}, Win rate: {trades_oos['win'].mean():.1%}")
else:
    print("No trades generados en out‑of‑sample.")

=== IN-SAMPLE (2020-2023) ===
Trades: 1110, Win rate: 48.3%

=== OUT-OF-SAMPLE (2024-2026) ===
Trades: 688, Win rate: 51.2%


In [11]:
# Celda 11: Monte Carlo de PnL diario (para estimar resultados en un día de torneo)
if len(trades) > 0:
    daily_pnl = trades.resample('D', on='entry_time')['pnl_points'].sum()
    n_sim = 10000
    sim_daily = np.random.choice(daily_pnl.values, size=n_sim, replace=True)
    sim_daily_usd = sim_daily * 15   # 3 MNQ = 15 USD por punto
    print("=== MONTE CARLO (10,000 días de torneo) ===")
    print(f"Mediana PnL diario: ${np.median(sim_daily_usd):,.0f}")
    print(f"Percentil 10: ${np.percentile(sim_daily_usd, 10):,.0f}")
    print(f"Percentil 90: ${np.percentile(sim_daily_usd, 90):,.0f}")
    print(f"Mejor día simulado: ${np.max(sim_daily_usd):,.0f}")
    print(f"Peor día simulado: ${np.min(sim_daily_usd):,.0f}")
    print(f"Probabilidad de pérdida diaria: {(sim_daily_usd < 0).mean():.1%}")
else:
    print("No hay trades para simular Monte Carlo.")

=== MONTE CARLO (10,000 días de torneo) ===
Mediana PnL diario: $0
Percentil 10: $0
Percentil 90: $0
Mejor día simulado: $4,538
Peor día simulado: $-1,510
Probabilidad de pérdida diaria: 1.8%


In [14]:
# Celda 14: Guardar trades a CSV para análisis posterior
if len(trades) > 0:
    trades.to_csv('tournament_trades_nq.csv', index=False)
    print("Trades guardados en 'tournament_trades_nq.csv'")
else:
    print("No hay trades para guardar.")

Trades guardados en 'tournament_trades_nq.csv'
